In [ ]:
# 03 - testing the pdf extraction on the fake statements from notebook 02
from pathlib import Path
import sys, re
import pandas as pd
import pdfplumber

# extraction + cleaning code lives in src/ (or this folder when running locally)
sys.path += ["../src", "."]
from pdf_extraction import extract_transactions, ExtractionError
from nlp_preprocessing import clean_description

IN_REPO = Path("../data/synthetic_statements/statements_index.csv").exists()
STMT_DIR = Path("../data/synthetic_statements") if IN_REPO else Path(r"C:\Users\This pc") / "finsight_statements"
index = pd.read_csv(STMT_DIR / "statements_index.csv")
print(f"{len(index)} statements found in {STMT_DIR}")
index[["statement_id", "bank_name", "layout", "period_start", "n_transactions", "pdf_file"]].head()

In [ ]:
# try one statement from each bank first
for f in index.groupby("bank")["pdf_file"].first():
    ex = extract_transactions(STMT_DIR / "pdfs" / f)
    print(f"{f}: {len(ex)} transactions | opening balance R{ex.attrs['opening_balance']:,.2f} | "
          f"balance check passed on {ex['balance_ok'].mean():.0%} of rows")
    display(ex.head(5))

In [ ]:
# why we didnt just use pdfplumber's extract_table()
# it needs table lines to find the columns, so check how many rows it gets per layout
baseline = []
for bank, g in index.groupby("bank"):
    r = g.iloc[0]
    found = 0
    with pdfplumber.open(STMT_DIR / "pdfs" / r.pdf_file) as pdf:
        for page in pdf.pages:
            for table in page.extract_tables():
                found += sum(1 for row in table[1:] if row and row[0] and re.match(r"\d", row[0]))
    expected = r.n_transactions + 1   # +1 for the opening balance row
    baseline.append({"bank": r.bank_name, "layout": r.layout,
                     "extract_table rows": found, "expected rows": expected,
                     "found %": round(100 * found / expected, 1)})
pd.DataFrame(baseline)

In [ ]:
# run the extractor on all 30 statements and compare with the ground truth
results, extracted_all = [], []
for r in index.itertuples():
    ex = extract_transactions(STMT_DIR / "pdfs" / r.pdf_file)
    gt = pd.read_csv(STMT_DIR / "ground_truth" / r.ground_truth_file)
    n = min(len(ex), len(gt))
    e, g = ex.iloc[:n].reset_index(drop=True), gt.iloc[:n]
    results.append({
        "statement_id": r.statement_id, "bank": r.bank_name, "layout": r.layout,
        "rows_expected": len(gt), "rows_extracted": len(ex),
        "date_acc": (e["date"].astype(str) == g["date"].astype(str)).mean(),
        "amount_acc": ((e["amount"] - g["amount"]).abs() < 0.005).mean(),
        "balance_acc": ((e["balance"] - g["balance"]).abs() < 0.005).mean(),
        "description_acc": (e["description_raw"] == g["description_pdf"]).mean(),
        # most important one: cleaned text must match what the model was trained on
        "clean_text_acc": (e["description_clean"] == g["description_sa"]).mean(),
        "balance_check": e["balance_ok"].mean(),
        "opening_ok": abs(ex.attrs["opening_balance"] - r.opening_balance) < 0.005,
    })
    extracted_all.append(ex.assign(statement_id=r.statement_id, bank=r.bank))

res = pd.DataFrame(results)
metrics = ["date_acc", "amount_acc", "balance_acc", "description_acc", "clean_text_acc", "balance_check"]
print(f"Rows: expected {res.rows_expected.sum():,} | extracted {res.rows_extracted.sum():,} | "
      f"opening balances correct: {res.opening_ok.sum()}/{len(res)}")
# accuracy % per bank + overall
summary = res.groupby(["bank", "layout"])[metrics].mean().mul(100).round(2).reset_index()
overall = pd.DataFrame([{"bank": "ALL", "layout": "", **res[metrics].mean().mul(100).round(2).to_dict()}])
summary = pd.concat([summary, overall], ignore_index=True)
summary

In [ ]:
# bad files should give a proper error message instead of crashing the app
from reportlab.pdfgen import canvas

tests_dir = STMT_DIR / "error_tests"
tests_dir.mkdir(exist_ok=True)
c = canvas.Canvas(str(tests_dir / "not_a_statement.pdf")); c.drawString(100, 700, "Dear customer, this is a letter."); c.save()
c = canvas.Canvas(str(tests_dir / "blank_like_a_scan.pdf")); c.showPage(); c.save()

for f in sorted(tests_dir.glob("*.pdf")):
    try:
        extract_transactions(f)
        print(f"{f.name}: extracted (unexpected)")
    except ExtractionError as err:
        print(f"{f.name}: rejected -> {err}")

In [ ]:
# save everything - extracted_all.csv is what the classifier will get from the pdfs
extracted = pd.concat(extracted_all, ignore_index=True)
extracted.to_csv(STMT_DIR / "extracted_all.csv", index=False)
res.to_csv(STMT_DIR / "extraction_evaluation.csv", index=False)
print(f"Saved extracted_all.csv ({len(extracted):,} rows) and extraction_evaluation.csv to {STMT_DIR}")
extracted.sample(8, random_state=3)[["statement_id", "date", "description_raw", "description_clean", "amount", "balance"]]